**IMPORTING DEPENDENCIES**

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

**LOADING CLEANED DATASET**

In [3]:
df = pd.read_csv(
    "../data/cleaned/supply_chain_cleaned.csv",
    parse_dates=["OrderDate", "ShippingDate"]
)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 41 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   OrderType              180519 non-null  object        
 1   ActualShippingDays     180519 non-null  int64         
 2   ScheduledShippingDays  180519 non-null  int64         
 3   Profit                 180519 non-null  float64       
 4   DeliveryStatus         180519 non-null  object        
 5   LateDeliveryRisk       180519 non-null  int64         
 6   CategoryID             180519 non-null  int64         
 7   CategoryName           180519 non-null  object        
 8   CustomerCity           180519 non-null  object        
 9   CustomerCountry        180519 non-null  object        
 10  CustomerFirstName      180519 non-null  object        
 11  CustomerID             180519 non-null  int64         
 12  CustomerLastName       180519 non-null  obje

**CREATING DATE FEATURES**

In [5]:
#order year
df["OrderYear"] = df["OrderDate"].dt.year

In [6]:
#order quarter
df["OrderQuarter"] = df["OrderDate"].dt.quarter

In [7]:
#order month number
df["OrderMonthNumber"] = df["OrderDate"].dt.month

In [8]:
#order month name
df["OrderMonth"] = df["OrderDate"].dt.month_name()

In [9]:
#day of week
df["OrderDay"] = df["OrderDate"].dt.day_name()

In [11]:
#weekend order (trur or false)
df["IsWeekend"] = df["OrderDate"].dt.dayofweek >= 5

**CREATING SHIPPING FEATURES**

In [13]:
#shipping delay
df["ShippingDelay"] = (df["ActualShippingDays"] - df["ScheduledShippingDays"])

In [14]:
#late delivery
df["IsLateDelivery"] = np.where(df["ShippingDelay"] > 0,"Yes","No")

In [19]:
#delivery performance
df["DeliveryPerformance"] = np.select(condlist=[df["ShippingDelay"] < 0, 
                                                df["ShippingDelay"] == 0, 
                                                df["ShippingDelay"] > 0], choicelist=["Early", "On Time", "Late"], default="Unknown")

**PROFIT FEATURES**

In [21]:
#profit margin percentage
df["ProfitMarginPercent"] = (df["Profit"] /df["GrossSales"]) * 100

In [22]:
#profit categories
df["ProfitCategory"] = pd.cut(df["Profit"],bins=[-1000,0,100,500,10000],labels=["Loss","Low Profit","Medium Profit","High Profit"])

**ORDER VALUE CATEGORY**

In [23]:
df["OrderValueCategory"] = pd.qcut(df["GrossSales"],q=4,labels=["Low","Medium","High","Premium"])

**DISCOUNT APPLIED(YES OR NO)**

In [24]:
df["HasDiscount"] = np.where(df["DiscountAmount"] > 0,"Yes","No")

In [26]:
#discount category
df["DiscountCategory"] = pd.cut(df["DiscountRate"],bins=[0,0.1,0.2,0.3,1],labels=["Low","Medium","High","Very High"],include_lowest=True)

**SHIPPING SPEED**

In [27]:
df["ShippingSpeed"] = pd.cut(df["ActualShippingDays"],bins=[0,2,4,10],labels=["Fast","Standard","Slow"])

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 56 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   OrderType              180519 non-null  object        
 1   ActualShippingDays     180519 non-null  int64         
 2   ScheduledShippingDays  180519 non-null  int64         
 3   Profit                 180519 non-null  float64       
 4   DeliveryStatus         180519 non-null  object        
 5   LateDeliveryRisk       180519 non-null  int64         
 6   CategoryID             180519 non-null  int64         
 7   CategoryName           180519 non-null  object        
 8   CustomerCity           180519 non-null  object        
 9   CustomerCountry        180519 non-null  object        
 10  CustomerFirstName      180519 non-null  object        
 11  CustomerID             180519 non-null  int64         
 12  CustomerLastName       180519 non-null  obje

In [29]:
df.head()

,OrderType,ActualShippingDays,ScheduledShippingDays,Profit,DeliveryStatus,LateDeliveryRisk,CategoryID,CategoryName,CustomerCity,CustomerCountry,CustomerFirstName,CustomerID,CustomerLastName,CustomerSegment,CustomerState,CustomerStreet,DepartmentID,DepartmentName,Latitude,Longitude,Market,OrderCity,OrderCountry,OrderDate,OrderID,DiscountAmount,DiscountRate,OrderItemID,ProfitRatio,Quantity,GrossSales,NetSales,OrderRegion,OrderState,OrderStatus,ProductID,ProductImage,ProductName,ProductPrice,ShippingDate,ShippingMode,OrderYear,OrderQuarter,OrderMonthNumber,OrderMonth,OrderDay,IsWeekend,ShippingDelay,IsLateDelivery,DeliveryPerformance,ProfitMarginPercent,ProfitCategory,OrderValueCategory,HasDiscount,DiscountCategory,ShippingSpeed
0,DEBIT,3,4,91.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,Cally,20755,Holloway,Consumer,PR,5365 Noble Nectar Island,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,2018-01-31 22:56:00,77202,13.110000,0.04,180517,0.29,1,327.75,314.640015,Southeast Asia,Java Occidental,COMPLETE,1360,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,2018-02-03 22:56:00,Standard Class,2018,1,1,January,Wednesday,False,-1,No,Early,27.841342,Low Profit,Premium,Yes,Low,Standard
1,TRANSFER,5,4,-249.089996,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,Irene,19492,Luna,Consumer,PR,2679 Rustic Loop,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,2018-01-13 12:27:00,75939,16.389999,0.05,179254,-0.80,1,327.75,311.359985,South Asia,Rajastán,PENDING,1360,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,2018-01-18 12:27:00,Standard Class,2018,1,1,January,Saturday,True,1,Yes,Late,-75.999999,Loss,Premium,Yes,Low,Slow
2,CASH,4,4,-247.779999,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,Gillian,19491,Maldonado,Consumer,CA,8510 Round Bear Gate,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,2018-01-13 12:06:00,75938,18.030001,0.06,179253,-0.80,1,327.75,309.720001,South Asia,Rajastán,CLOSED,1360,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,2018-01-17 12:06:00,Standard Class,2018,1,1,January,Saturday,True,0,No,On Time,-75.600305,Loss,Premium,Yes,Low,Standard
3,DEBIT,3,4,22.860001,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,Tana,19490,Tate,Home Office,CA,3200 Amber Bend,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,2018-01-13 11:45:00,75937,22.940001,0.07,179252,0.08,1,327.75,304.809998,Oceania,Queensland,COMPLETE,1360,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,2018-01-16 11:45:00,Standard Class,2018,1,1,January,Saturday,True,-1,No,Early,6.974829,Low Profit,Premium,Yes,Low,Standard
4,PAYMENT,2,4,134.210007,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,Orli,19489,Hendricks,Corporate,PR,8671 Iron Anchor Corners,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,2018-01-13 11:24:00,75936,29.500000,0.09,179251,0.45,1,327.75,298.250000,Oceania,Queensland,PENDING_PAYMENT,1360,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,2018-01-15 11:24:00,Standard Class,2018,1,1,January,Saturday,True,-2,No,Early,40.948896,Medium Profit,Premium,Yes,Low,Fast


In [30]:
df.to_csv("../data/cleaned/supply_chain_features.csv",index=False)